In [6]:
import pandas as pd
import numpy as np
from snowflake.snowpark import Session


In [8]:
connection_parameters = {
    "account": "IJYGGJU-WI05238",
    "user": "Shank227",
    "password": "ShashankB2207$",
    "warehouse": "COMPUTE_WH",
    "database": "BANK_SEGMENTATION_DB",
    "schema": "PUBLIC",
    "role": "ACCOUNTADMIN"
}

In [9]:
session = Session.builder.configs(connection_parameters).create()

In [10]:
cluster_df = session.table("CUSTOMER_CLUSTERS").to_pandas()

In [13]:
cluster_df.columns

Index(['AGE', 'DURATION', 'CAMPAIGN', 'PDAYS', 'PREVIOUS', 'EMPLOYMENT_RATE',
       'CONSUMER_PRICE_INDEX', 'CONSUMER_CONFIDENCE_INDEX', 'EURIBOR3M',
       'EMPLOYEES_COUNT', 'JOB_BLUE_COLLAR', 'JOB_ENTREPRENEUR',
       'JOB_HOUSEMAID', 'JOB_MANAGEMENT', 'JOB_RETIRED', 'JOB_SELFEMPLOYED',
       'JOB_SERVICES', 'JOB_STUDENT', 'JOB_TECHNICIAN', 'JOB_UNEMPLOYED',
       'JOB_UNKNOWN', 'MARITAL_MARRIED', 'MARITAL_SINGLE', 'MARITAL_UNKNOWN',
       'EDUCATION_BASIC6Y', 'EDUCATION_BASIC9Y', 'EDUCATION_HIGHSCHOOL',
       'EDUCATION_ILLITERATE', 'EDUCATION_PROFESSIONALCOURSE',
       'EDUCATION_UNIVERSITYDEGREE', 'EDUCATION_UNKNOWN', 'DEFAULT_UNKNOWN',
       'DEFAULT_YES', 'HOUSING_UNKNOWN', 'HOUSING_YES', 'LOAN_UNKNOWN',
       'LOAN_YES', 'CONTACT_TELEPHONE', 'MONTH_AUG', 'MONTH_DEC', 'MONTH_JUL',
       'MONTH_JUN', 'MONTH_MAR', 'MONTH_MAY', 'MONTH_NOV', 'MONTH_OCT',
       'MONTH_SEP', 'DAY_OF_WEEK_MON', 'DAY_OF_WEEK_THU', 'DAY_OF_WEEK_TUE',
       'DAY_OF_WEEK_WED', 'POUTCOME_NONEXI

In [18]:
df_encoded = pd.read_csv("encoded_dataset.csv")
df_encoded.head()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False


In [19]:
df_encoded["Cluster"] = cluster_df["Cluster"]

In [24]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 54 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            41188 non-null  int64  
 1   duration                       41188 non-null  int64  
 2   campaign                       41188 non-null  int64  
 3   pdays                          41188 non-null  int64  
 4   previous                       41188 non-null  int64  
 5   emp.var.rate                   41188 non-null  float64
 6   cons.price.idx                 41188 non-null  float64
 7   cons.conf.idx                  41188 non-null  float64
 8   euribor3m                      41188 non-null  float64
 9   nr.employed                    41188 non-null  float64
 10  job_blue-collar                41188 non-null  bool   
 11  job_entrepreneur               41188 non-null  bool   
 12  job_housemaid                  41188 non-null 

In [25]:
df_encoded["Cluster"].value_counts()

Cluster
0    24071
2    11492
1     5625
Name: count, dtype: int64

In [26]:
numerical_profile = df_encoded.groupby("Cluster")[
    [
        "age",
        "duration",
        "campaign",
        "pdays",
        "previous"
    ]
].mean().round(2)

numerical_profile

,age,duration,campaign,pdays,previous
Cluster,,,,,
0,40.09,252.39,2.78,992.89,0.08
1,40.44,269.88,2.32,921.51,0.28
2,39.68,264.97,2.24,918.83,0.32


In [27]:
economic_profile = df_encoded.groupby("Cluster")[
    [
        "emp.var.rate",
        "cons.price.idx",
        "cons.conf.idx",
        "euribor3m",
        "nr.employed"
    ]
].mean().round(2)

economic_profile

,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
Cluster,,,,,
0,0.65,93.66,-41.63,4.26,5201.99
1,-0.37,93.60,-38.04,3.13,5127.85
2,-0.89,93.39,-39.34,2.53,5113.01


In [28]:
engineered_profile = df_encoded.groupby("Cluster")[
    [
        "age_group",
        "contacted_before",
        "high_campaign"
    ]
].mean().round(2)

engineered_profile

KeyError: "Columns not found: 'age_group', 'contacted_before', 'high_campaign'"

In [29]:
job_columns = [col for col in df_encoded.columns if col.startswith("job_")]

job_profile = df_encoded.groupby("Cluster")[job_columns].mean().T

job_profile

Cluster,0,1,2
job_blue-collar,0.217149,0.234311,0.235729
job_entrepreneur,0.039217,0.029156,0.030282
job_housemaid,0.027211,0.026844,0.022102
job_management,0.073491,0.069156,0.066655
job_retired,0.031947,0.058311,0.054212
job_self-employed,0.036849,0.028800,0.032370
job_services,0.092560,0.100089,0.102506
job_student,0.010801,0.031467,0.038113
job_technician,0.182336,0.141333,0.135660
job_unemployed,0.023638,0.023467,0.027236


In [30]:
education_columns = [col for col in df_encoded.columns if col.startswith("education_")]

education_profile = df_encoded.groupby("Cluster")[education_columns].mean().T

education_profile

Cluster,0,1,2
education_basic.6y,0.052885,0.058311,0.060129
education_basic.9y,0.141415,0.151644,0.155586
education_high.school,0.221927,0.238933,0.246171
education_illiterate,0.000499,0.000533,0.000261
education_professional.course,0.135516,0.113778,0.116690
education_university.degree,0.311703,0.272711,0.272450
education_unknown,0.039010,0.045689,0.046554


In [31]:
marital_columns = [col for col in df_encoded.columns if col.startswith("marital_")]

marital_profile = df_encoded.groupby("Cluster")[marital_columns].mean().T

marital_profile

Cluster,0,1,2
marital_married,0.617590,0.593600,0.585016
marital_single,0.264592,0.292800,0.309085
marital_unknown,0.001828,0.002489,0.001914
